# Feature Engineering


In [37]:
import numpy as np
import pandas as pd
import yfinance as yf
from market_ml.data_loader import DataLoader
import matplotlib.pyplot as plt
import plotly.express as px
from market_ml.config import * 
from market_ml.EDA import * 
from market_ml.feature_engineering import *
from market_ml.models import *

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import *
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit


# Data Loading 


In [38]:
stock_dl = DataLoader(tickers=TICKERS, period="10y", interval="1d", file_name='stock_data.parquet')
benchmark_dl = DataLoader(tickers=[BENCHMARK_TICKERS], period="10y", interval="1d", file_name='benchmark_data.parquet')

market_data = stock_dl.load_data()
benchmark_data = benchmark_dl.load_data()
sectors = market_data["Sector"].unique()

benchmark_data

Data loaded for ['AAPL', 'MSFT', 'NVDA', 'META', 'GOOGL', 'JPM', 'GS', 'BAC', 'MS', 'BLK', 'JNJ', 'UNH', 'ABBV', 'PFE', 'MRK', 'XOM', 'CVX', 'COP', 'SLB', 'EOG', 'AMZN', 'COST', 'WMT', 'HD', 'MCD']
Data loaded for [['SPY', 'QQQ']]


Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-09-29,QQQ,NaN,110.121559,110.943924,109.644969,110.738333,27207500
1,2016-09-29,SPY,NaN,183.335754,185.206007,182.789198,184.804628,128070600
2,2016-09-30,QQQ,NaN,110.943939,111.271013,110.299131,110.411274,25543200
3,2016-09-30,SPY,NaN,184.719223,185.419493,183.916465,184.164118,117202900
4,2016-10-03,QQQ,NaN,110.785042,110.925211,110.355165,110.803729,19216400
...,...,...,...,...,...,...,...,...
5021,2026-09-25,SPY,NaN,771.349976,772.280029,766.289978,768.780029,36666700
5022,2026-09-28,QQQ,NaN,736.530029,741.419983,731.630005,740.460022,41716200
5023,2026-09-28,SPY,NaN,765.609985,769.539978,763.719971,768.349976,42296700
5024,2026-09-29,QQQ,NaN,737.229980,740.579895,736.000000,740.169983,9703756


In [39]:
#just testing with one stock for now 

test_ticker = ['AAPL', 'NVDA']
test_data = market_data[market_data["Ticker"].isin(test_ticker)].copy()

test_data

Price,Date,Ticker,Sector,Close,High,Low,Open,Volume
0,2016-09-29,AAPL,Technology,25.653965,26.024437,25.567065,25.878078,143548000
19,2016-09-29,NVDA,Technology,1.653514,1.658666,1.620394,1.635605,336684000
25,2016-09-30,AAPL,Technology,25.852909,25.926088,25.567052,25.717984,145516400
44,2016-09-30,NVDA,Technology,1.680990,1.697427,1.652778,1.662591,429932000
50,2016-10-03,AAPL,Technology,25.731701,25.852906,25.676817,25.775152,86807200
...,...,...,...,...,...,...,...,...
62769,2026-09-25,NVDA,Technology,225.070007,226.940002,223.130005,225.130005,89947700
62775,2026-09-28,AAPL,Technology,338.399994,342.989990,338.040009,340.369995,32724700
62794,2026-09-28,NVDA,Technology,228.860001,233.210007,228.039993,229.750000,141544300
62800,2026-09-29,AAPL,Technology,332.345001,337.059998,331.510010,337.059998,12025955


In [40]:
engineer = FeatureEngineer(test_data, benchmark_data, filename="test_stock_data_with_features.parquet")
data  = engineer.load_features()

data.shape

Data loaded for Price       Date Ticker      Sector       Close        High         Low  \
0     2016-09-29   AAPL  Technology   25.653965   26.024437   25.567065   
19    2016-09-29   NVDA  Technology    1.653514    1.658666    1.620394   
25    2016-09-30   AAPL  Technology   25.852909   25.926088   25.567052   
44    2016-09-30   NVDA  Technology    1.680990    1.697427    1.652778   
50    2016-10-03   AAPL  Technology   25.731701   25.852906   25.676817   
...          ...    ...         ...         ...         ...         ...   
62769 2026-09-25   NVDA  Technology  225.070007  226.940002  223.130005   
62775 2026-09-28   AAPL  Technology  338.399994  342.989990  338.040009   
62794 2026-09-28   NVDA  Technology  228.860001  233.210007  228.039993   
62800 2026-09-29   AAPL  Technology  332.345001  337.059998  331.510010   
62819 2026-09-29   NVDA  Technology  230.899994  232.820007  229.220001   

Price        Open     Volume  
0       25.878078  143548000  
19       1.635605  33

(4908, 21)

In [41]:
print(data.shape)

print(data[["Date", "Ticker"]].head())

print(data.sort_values(["Date", "Ticker"]).reset_index(drop=True))



(4908, 21)
Price       Date Ticker
98    2016-12-08   AAPL
99    2016-12-08   NVDA
100   2016-12-09   AAPL
101   2016-12-09   NVDA
102   2016-12-12   AAPL
Price       Date Ticker      Sector       Close        High         Low  \
0     2016-12-08   AAPL  Technology   25.771877   25.843133   25.422489   
1     2016-12-08   NVDA  Technology    2.296763    2.373912    2.286199   
2     2016-12-09   AAPL  Technology   26.192514   26.364909   25.815545   
3     2016-12-09   NVDA  Technology    2.255978    2.316174    2.228706   
4     2016-12-12   AAPL  Technology   26.043110   26.433871   25.856922   
...          ...    ...         ...         ...         ...         ...   
4903  2026-09-11   NVDA  Technology  218.289993  222.000000  218.149994   
4904  2026-09-14   AAPL  Technology  333.079987  335.500000  331.339996   
4905  2026-09-14   NVDA  Technology  210.960007  212.770004  208.929993   
4906  2026-09-15   AAPL  Technology  331.339996  331.779999  328.350006   
4907  2026-09-15   N

In [42]:
print("Start:", data["Date"].min())
print("End:  ", data["Date"].max())

Start: 2016-12-08 00:00:00
End:   2026-09-15 00:00:00


# Splitting Data into Train and Test 

In [43]:
data = data.drop(columns=["Target_5D"]) #removing the target column here as we don't need it as we're doing regression. 
train_data, test_data, split_date = split_chronological_data(data)



In [44]:
data.columns

Index(['Date', 'Ticker', 'Sector', 'Close', 'High', 'Low', 'Open', 'Volume',
       'QQQ_Return', 'SPY_Return', 'Daily_Return', 'Return_5D',
       'Volatility_20D', 'SMA_20D', 'SMA_50D', 'Price_to_SMA_20D',
       'Price_to_SMA_50D', 'Volume_Average_20D', 'Relative_Volume_20D',
       'Future_Return_5D'],
      dtype='str', name='Price')

In [45]:
features = list(train_data.columns)[:-1]
target = list(train_data)[-1]
features = ["QQQ_Return","SPY_Return","Daily_Return","Return_5D","Volatility_20D","Price_to_SMA_20D","Price_to_SMA_50D","Relative_Volume_20D",
] #want to remove everything with future information

In [46]:
x_train, y_train = train_data[features], train_data[target]
x_test, y_test = test_data[features], test_data[target]

x_test.columns

Index(['QQQ_Return', 'SPY_Return', 'Daily_Return', 'Return_5D',
       'Volatility_20D', 'Price_to_SMA_20D', 'Price_to_SMA_50D',
       'Relative_Volume_20D'],
      dtype='str', name='Price')

# Linear Regression

Not using standard scaler initially  

In [47]:


linearModel = LinearRegression()

linearModel.fit(x_train, y_train)

y_pred = linearModel.predict(x_test)

In [69]:
model_comparison_df = pd.DataFrame(index=["MAE", "RMSE", "R2"])

MAE = mean_absolute_error(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)

print(f"Linear Regression Mean absolute error is {np.round(MAE,4)}")
print(f"Linear Regression Root mean squared error is {np.round(RMSE,4)}")
print(f"Linear Regression R-squared is {np.round(R2,4)}")

linear_regression_results = [np.round(MAE,4), np.round(RMSE,4), np.round(R2, 4)]

model_comparison_df['Linear Regression'] = linear_regression_results

model_comparison_df

Linear Regression Mean absolute error is 0.0383
Linear Regression Root mean squared error is 0.0506
Linear Regression R-squared is -0.0111


,Linear Regression
MAE,0.0383
RMSE,0.0506
R2,-0.0111


In [ ]:
#let's create a baseline prediction -- just looking at the mean.

baseline_prediction = y_train.mean()
baseline_y_pred = [baseline_prediction] * len(y_test)
print(f"Baseline prediction: {np.round(baseline_prediction,4)}")

Baseline prediction: 0.0094


In [75]:
baseline_mae = mean_absolute_error(y_test, baseline_y_pred)
baseline_rmse = root_mean_squared_error(y_test, baseline_y_pred)
baseline_r2 = r2_score(y_test, baseline_y_pred)

print(f"Baseline Mean absolute error is {np.round(baseline_mae,4)}")
print(f"Baseline Root mean squared error is {np.round(baseline_rmse,4)}")
print(f"Baseline R-squared is {np.round(baseline_r2,4)}")

baseline_metrics = [np.round(baseline_mae,4), np.round(baseline_rmse,4), np.round(baseline_r2,4)]

model_comparison_df['Baseline Model'] = baseline_metrics

model_comparison_df = model_comparison_df[["Baseline Model", "Linear Regression"]]

model_comparison_df

Baseline Mean absolute error is 0.0381
Baseline Root mean squared error is 0.0504
Baseline R-squared is -0.0034


,Baseline Model,Linear Regression
MAE,0.0381,0.0383
RMSE,0.0504,0.0506
R2,-0.0034,-0.0111


Looks like our initial linear regression is pretty close to baseline. 

# Random Forest

In [53]:
ntrees = 100
max_depth = 5 #default is None which I don't want to use at it'll overfit
random_state = 42 #for reproducability 
njobs = -1 #allowing it to use all available cores on my computer 
rf_model = RandomForestRegressor(n_estimators=ntrees, max_depth=max_depth, random_state=random_state, n_jobs=njobs)

In [55]:
rf_model.fit(x_train, y_train)
y_pred = rf_model.predict(x_test)



In [77]:
MAE = mean_absolute_error(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)

print(f"Mean absolute error is {np.round(MAE,4)}")
print(f"Root mean squared error is {np.round(RMSE,4)}")
print(f"R-squared is {np.round(R2,4)}")

random_forest_metrics = [np.round(MAE,4), np.round(RMSE,4), np.round(R2, 4)]

model_comparison_df['Random Forest'] = random_forest_metrics

model_comparison_df

Mean absolute error is 0.0383
Root mean squared error is 0.0506
R-squared is -0.0111


,Baseline Model,Linear Regression,Random Forest
MAE,0.0381,0.0383,0.0383
RMSE,0.0504,0.0506,0.0506
R2,-0.0034,-0.0111,-0.0111


# Random Forest Hyperparameter Tuning

In [59]:
rf_Model = RandomForestRegressor(random_state=42)

cv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "n_estimators":[50, 100, 200, 300, 400],
    "max_depth": [3, 5, 10, 15, 20],
    "min_samples_leaf": [1, 2, 5],
}

grid_search = GridSearchCV(rf_Model,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)



In [60]:
grid_search.fit(x_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

{'max_depth': 3, 'min_samples_leaf': 1, 'n_estimators': 200}
-0.0426203431747558


In [65]:
best_parameters = grid_search.best_params_

rf_Model = RandomForestRegressor(n_estimators=best_parameters["n_estimators"], max_depth=best_parameters["max_depth"], min_samples_leaf=best_parameters["min_samples_leaf"], random_state=42)

rf_Model.fit(x_train, y_train)
y_pred = rf_Model.predict(x_test)




In [78]:
MAE = mean_absolute_error(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)

print(f"Tuned Random Forest Mean absolute error is {np.round(MAE,4)}")
print(f"Tuned Random Forest Root mean squared error is {np.round(RMSE,4)}")
print(f"Tuned Random R-squared is {np.round(R2,4)}")

tuned_random_forest_metrics = [np.round(MAE,4), np.round(RMSE, 4), np.round(R2, 4)]

model_comparison_df['Tuned Random Forest'] = tuned_random_forest_metrics

model_comparison_df

Tuned Random Forest Mean absolute error is 0.0383
Tuned Random Forest Root mean squared error is 0.0506
Tuned Random R-squared is -0.0111


,Baseline Model,Linear Regression,Random Forest,Tuned Random Forest
MAE,0.0381,0.0383,0.0383,0.0383
RMSE,0.0504,0.0506,0.0506,0.0506
R2,-0.0034,-0.0111,-0.0111,-0.0111


Random forest is worse compared to linear regression and mabseline model. Either something is wrong with the data prep or something else is going on here -- need to investigate the features. 